In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

print("All imports done!")
print("GPU available:", torch.cuda.is_available())

All imports done!
GPU available: True


In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

print("All imports done!")
print("GPU available:", torch.cuda.is_available())

All imports done!
GPU available: True


In [3]:
def normalize_hu(image, min_hu=-1000, max_hu=1000):
    image = np.clip(image, min_hu, max_hu)
    image = (image - min_hu) / (max_hu - min_hu)
    return image.astype(np.float32)

def get_patches(ldct, ndct, patch_size=64, n_patches=4):
    h, w = ldct.shape
    patches_ld, patches_nd = [], []
    for _ in range(n_patches):
        top  = np.random.randint(0, h - patch_size)
        left = np.random.randint(0, w - patch_size)
        patches_ld.append(ldct[top:top+patch_size, left:left+patch_size])
        patches_nd.append(ndct[top:top+patch_size, left:left+patch_size])
    return np.array(patches_ld), np.array(patches_nd)

print("Utilities defined!")

Utilities defined!


In [4]:
class REDCNN(nn.Module):
    def __init__(self, in_channels=1, num_filters=96, num_layers=5):
        super().__init__()
        self.num_layers = num_layers

        # Encoder: 5 Conv layers
        self.encoder_convs = nn.ModuleList()
        for i in range(num_layers):
            inc = in_channels if i == 0 else num_filters
            self.encoder_convs.append(
                nn.Conv2d(inc, num_filters, kernel_size=5, stride=1, padding=0)
            )

        # Decoder: 5 ConvTranspose layers
        self.decoder_convs = nn.ModuleList()
        for i in range(num_layers):
            outc = 1 if i == num_layers - 1 else num_filters
            self.decoder_convs.append(
                nn.ConvTranspose2d(num_filters, outc, kernel_size=5, stride=1, padding=0)
            )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        encoder_outputs = []

        # Encoding
        feat = x
        for conv in self.encoder_convs:
            feat = self.relu(conv(feat))
            encoder_outputs.append(feat)

        # Decoding with symmetric skip connections
        for i, deconv in enumerate(self.decoder_convs):
            skip_idx = self.num_layers - 1 - i
            feat = feat + encoder_outputs[skip_idx]   # shortcut
            feat = deconv(feat)
            if i != self.num_layers - 1:
                feat = self.relu(feat)

        return feat + residual   # global residual

# Quick test
model = REDCNN()
dummy = torch.randn(1, 1, 64, 64)
out = model(dummy)
print("Model output shape:", out.shape)   # should be (1, 1, 64, 64)
print("Total parameters:", sum(p.numel() for p in model.parameters()))

Model output shape: torch.Size([1, 1, 64, 64])
Total parameters: 1848865


In [5]:
class REDCNNLoss(nn.Module):
    """MSE loss — same as original RED-CNN paper."""
    def forward(self, pred, target):
        return F.mse_loss(pred, target)

print("Loss defined!")

Loss defined!


In [6]:
class CTDataset(Dataset):
    def __init__(self, ldct_list, ndct_list, train=True):
        self.ldct_list = ldct_list
        self.ndct_list = ndct_list
        self.train = train

    def __len__(self):
        return len(self.ldct_list)

    def __getitem__(self, idx):
        ld_img = Image.open(self.ldct_list[idx]).convert('L')
        nd_img = Image.open(self.ndct_list[idx]).convert('L')

        ld = np.array(ld_img).astype(np.float32) / 255.0
        nd = np.array(nd_img).astype(np.float32) / 255.0

        if self.train:
            ld_patches, nd_patches = get_patches(ld, nd)
            i = np.random.randint(0, 4)
            return (torch.from_numpy(ld_patches[i]).unsqueeze(0),
                    torch.from_numpy(nd_patches[i]).unsqueeze(0))

        return torch.from_numpy(ld).unsqueeze(0), torch.from_numpy(nd).unsqueeze(0)

print("Dataset class defined!")

Dataset class defined!


In [7]:
def collect_image_paths(root_dir, slice_thickness="1mm", kernel="Soft Kernel (B30)"):
    ld_root = os.path.join(root_dir, "Quarter Dose", slice_thickness, kernel)
    nd_root = os.path.join(root_dir, "Full Dose",   slice_thickness, kernel)
    ld_files, nd_files = [], []
    for patient in sorted(os.listdir(ld_root)):
        ld_p = os.path.join(ld_root, patient)
        nd_p = os.path.join(nd_root, patient)
        for ls, ns in zip(sorted(os.listdir(ld_p)), sorted(os.listdir(nd_p))):
            ld_files.append(os.path.join(ld_p, ls))
            nd_files.append(os.path.join(nd_p, ns))
    return ld_files, nd_files

# ---- Change this path if your dataset folder looks different ----
data_root = "/kaggle/input/datasets/andrewmvd/ct-low-dose-reconstruction/Preprocessed_512x512/512"

ldct_files, ndct_files = collect_image_paths(data_root)
print("Total image pairs:", len(ldct_files))

# 90% train / 10% test split
split = int(0.9 * len(ldct_files))
train_ds = CTDataset(ldct_files[:split], ndct_files[:split], train=True)
test_ds  = CTDataset(ldct_files[split:], ndct_files[split:], train=False)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=1,  shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)} | Test images: {len(test_ds)}")

Total image pairs: 5936
Train batches: 334 | Test images: 594


In [8]:
def train_model(model, train_loader, epochs=100):   # start with 100, increase later
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = REDCNNLoss().to(device)

    history = {'loss': [], 'psnr': [], 'ssim': []}
    print(f"Training RED-CNN on {device} for {epochs} epochs...\n")

    for epoch in range(epochs):
        model.train()
        total_loss = total_psnr = total_ssim = 0.0

        for ld, nd in train_loader:
            ld, nd = ld.to(device), nd.to(device)
            optimizer.zero_grad()
            output = model(ld)
            loss = criterion(output, nd)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            with torch.no_grad():
                p_np = output.clamp(0, 1).cpu().numpy()
                t_np = nd.cpu().numpy()
                bs = p_np.shape[0]
                bp = bs_ = 0.0
                for i in range(bs):
                    p = p_np[i].squeeze(); t = t_np[i].squeeze()
                    bp  += psnr(t, p, data_range=1.0)
                    bs_ += ssim(t, p, data_range=1.0)
                total_psnr += bp / bs
                total_ssim += bs_ / bs

        n = len(train_loader)
        history['loss'].append(total_loss / n)
        history['psnr'].append(total_psnr / n)
        history['ssim'].append(total_ssim / n)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}]"
                  f"  Loss: {history['loss'][-1]:.6f}"
                  f"  PSNR: {history['psnr'][-1]:.2f} dB"
                  f"  SSIM: {history['ssim'][-1]:.4f}")

    torch.save(model.state_dict(), "redcnn_weights.pth")
    print("\nModel saved as redcnn_weights.pth")
    return history

model   = REDCNN()
history = train_model(model, train_loader, epochs=100)

Training RED-CNN on cuda for 100 epochs...

Epoch [10/100]  Loss: 0.000494  PSNR: 34.55 dB  SSIM: 0.8764
Epoch [20/100]  Loss: 0.000364  PSNR: 35.84 dB  SSIM: 0.8920
Epoch [30/100]  Loss: 0.000258  PSNR: 37.27 dB  SSIM: 0.9107
Epoch [40/100]  Loss: 0.000231  PSNR: 37.78 dB  SSIM: 0.9169
Epoch [50/100]  Loss: 0.000212  PSNR: 38.24 dB  SSIM: 0.9235
Epoch [60/100]  Loss: 0.000199  PSNR: 38.55 dB  SSIM: 0.9270
Epoch [70/100]  Loss: 0.000197  PSNR: 38.68 dB  SSIM: 0.9293
Epoch [80/100]  Loss: 0.000196  PSNR: 38.69 dB  SSIM: 0.9277
Epoch [90/100]  Loss: 0.000177  PSNR: 39.18 dB  SSIM: 0.9330
Epoch [100/100]  Loss: 0.000184  PSNR: 39.02 dB  SSIM: 0.9301

Model saved as redcnn_weights.pth


In [9]:
import os

print(os.listdir())

['.virtual_documents', 'redcnn_weights.pth']


In [10]:
model = REDCNN()
model.load_state_dict(torch.load("redcnn_weights.pth"))

<All keys matched successfully>

In [11]:
history2 = train_model(model, train_loader, epochs=100)

Training RED-CNN on cuda for 100 epochs...

Epoch [10/100]  Loss: 0.000189  PSNR: 38.92 dB  SSIM: 0.9304
Epoch [20/100]  Loss: 0.000175  PSNR: 39.26 dB  SSIM: 0.9325
Epoch [30/100]  Loss: 0.000178  PSNR: 39.17 dB  SSIM: 0.9329
Epoch [40/100]  Loss: 0.000165  PSNR: 39.45 dB  SSIM: 0.9345
Epoch [50/100]  Loss: 0.000171  PSNR: 39.41 dB  SSIM: 0.9342
Epoch [60/100]  Loss: 0.000171  PSNR: 39.39 dB  SSIM: 0.9343
Epoch [70/100]  Loss: 0.000160  PSNR: 39.67 dB  SSIM: 0.9362
Epoch [80/100]  Loss: 0.000167  PSNR: 39.42 dB  SSIM: 0.9340
Epoch [90/100]  Loss: 0.000163  PSNR: 39.59 dB  SSIM: 0.9352
Epoch [100/100]  Loss: 0.000150  PSNR: 39.94 dB  SSIM: 0.9382

Model saved as redcnn_weights.pth


In [13]:
torch.save(model.state_dict(), "redcnn_200epochs.pth")

In [14]:
import os
print(os.listdir())

['.virtual_documents', 'redcnn_200epochs.pth', 'redcnn_weights.pth']


In [ ]:
def evaluate(model, test_loader):
    model.eval()
    device = next(model.parameters()).device
    psnr_vals, ssim_vals = [], []
    with torch.no_grad():
        for ld, nd in test_loader:
            ld = ld.to(device)
            pred   = model(ld).clamp(0, 1).cpu().squeeze().numpy()
            target = nd.squeeze().numpy()
            psnr_vals.append(psnr(target, pred, data_range=1.0))
            ssim_vals.append(ssim(target, pred, data_range=1.0))
    print(f"Test PSNR : {np.mean(psnr_vals):.2f} dB")
    print(f"Test SSIM : {np.mean(ssim_vals):.4f}")

evaluate(model, test_loader)